In [ ]:
! pip install openai gradio tavily-python

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from tavily import TavilyClient
import gradio as gr
import json

In [ ]:
load_dotenv(override=True)

api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key:
    raise ValueError("OPENROUTER_API_KEY not found in .env file")


TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
if not TAVILY_API_KEY:
    raise ValueError("TAVILY_API_KEY not found in .env file")

tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

# OpenAI client
openai_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)

# Ollama client (OpenAI-compatible endpoint)
ollama_client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

In [ ]:
MODEL_CHOICES = ["gpt-4.1-mini", "gpt-4o", "gpt-4-turbo"]
openai = OpenAI()


In [ ]:
system_message = """
You are ChatGPT, a helpful, knowledgeable, and reliable AI assistant.

You provide clear, accurate, and well-structured answers.
When needed, you use available tools to retrieve up-to-date or external information.
If you don't know something, you say so honestly.
You explain your reasoning clearly and help users learn step-by-step.
"""

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "tavily_search",
            "description": "Search the web for up-to-date information using Tavily.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query to look up on the web"
                    }
                },
                "required": ["query"],
            },
        },
    },
]

In [ ]:
def tavily_search(query):
    if not query:
        return "No query provided."

    response = tavily_client.search(
        query=query,
        search_depth="advanced",
        include_answer=True,
        max_results=5,
    )

    if response.get("answer"):
        return response["answer"]

    results = response.get("results", [])
    if not results:
        return "No results found."

    combined = "\n\n".join(
        f"{r.get('title')}\n{r.get('content')}"
        for r in results
    )

    return combined

In [ ]:

def accumulate_stream(stream):
    content = ""
    tool_calls_by_index = {}
    finish_reason = None

    for chunk in stream:
        if not chunk.choices:
            continue

        delta = chunk.choices[0].delta

        if getattr(delta, "content", None):
            content += delta.content or ""

        if getattr(delta, "tool_calls", None):
            for tc in delta.tool_calls:
                idx = tc.index

                if idx not in tool_calls_by_index:
                    tool_calls_by_index[idx] = {
                        "id": "",
                        "type": "function",
                        "function": {"name": "", "arguments": ""}
                    }

                if tc.id:
                    tool_calls_by_index[idx]["id"] = tc.id

                if getattr(tc.function, "name", None):
                    tool_calls_by_index[idx]["function"]["name"] = tc.function.name

                if getattr(tc.function, "arguments", None):
                    tool_calls_by_index[idx]["function"]["arguments"] += (
                        tc.function.arguments or ""
                    )

        if chunk.choices[0].finish_reason:
            finish_reason = chunk.choices[0].finish_reason

    tool_calls_list = (
        [tool_calls_by_index[i] for i in sorted(tool_calls_by_index)]
        if tool_calls_by_index else None
    )

    return content, tool_calls_list, finish_reason

In [ ]:
def chat(message, history, model):
    messages = [{"role": "system", "content": system_message}]

    for h in history:
        messages.append({"role": "user", "content": h["content"]}) if h["role"] == "user" \
            else messages.append({"role": "assistant", "content": h["content"]})

    messages.append({"role": "user", "content": message})

    accumulated_text = ""

    while True:
        stream = openai.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools,
            stream=True,
        )

        content, tool_calls_list, finish_reason = accumulate_stream(stream)

        accumulated_text += content
        yield accumulated_text

        if finish_reason != "tool_calls" or not tool_calls_list:
            break

        # Add assistant tool call message
        assistant_msg = {
            "role": "assistant",
            "content": content or None,
            "tool_calls": tool_calls_list,
        }
        messages.append(assistant_msg)

        # Execute tools
        tool_responses = []

        for tc in tool_calls_list:
            fn_name = tc["function"]["name"]
            args_str = tc["function"]["arguments"]

            if fn_name == "tavily_search":
                args = json.loads(args_str) if args_str else {}
                result = tavily_search(args.get("query", ""))

                tool_responses.append({
                    "role": "tool",
                    "tool_call_id": tc["id"],
                    "content": result,
                })

        messages.extend(tool_responses)


In [ ]:
model_dropdown = gr.Dropdown(
    choices=MODEL_CHOICES,
    value=MODEL_CHOICES[0],
    label="Model",
)

gr.ChatInterface(
    fn=chat,
    type="messages",
    additional_inputs=[model_dropdown],
).launch()